# Fine-Tuning Qwen2.5-0.5B for Ecosystem Intent, Skill Routing & Task Time Allocation

This notebook fine-tunes **`Qwen2.5-0.5B-Instruct`** using **Unsloth** and LoRA to serve as the ultra-fast local router for `adaptive-workflow` and `Fleet-Orchestrator`.

### Capabilities Trained:
1. **Lifecycle Archetype & Skill Matrix Routing** (`ENGINEERING_DEV`, `RESEARCH_ACADEMIC`, `DOMAIN_HARDWARE`, `FRONTEND_PRODUCT`, `SWARM_ORCHESTRATION`, `SYSADMIN_SECURITY`).
2. **Task Time Allocation**:
   - `tier`: Discrete time complexity tier (`T0_MICRO`, `T1_FAST`, `T2_MEDIUM`, `T3_LONG`, `T4_EPIC`)
   - `estimated_duration_s`: Empirical wall-clock duration estimate
   - `timeout_ceiling_s`: Dynamic worker timeout ceiling for `copilot_queue_worker.py`
   - `cpm_weight`: Normalized duration factor ($D_j$) for mathematical Critical Path Method scheduling in `sim/adaptive_orchestrator.py`
   - `execution_route`: Dynamic routing target (`DIRECT_FAST`, `SUBAGENT`, `FLEET_WORKER`, `HUMAN_CONFIRM`)

### Runtime Setting (Colab Pro):
- **Hardware Accelerator**: Select **L4 GPU** (Ada Lovelace, 24GB VRAM) or **T4 GPU**.
- Total training time on L4: **~60 to 90 seconds** (for 1,553 samples).
- Output: Quantized **`q4_k_m` GGUF** model file (~380 MB).

In [ ]:
# 1. Install Unsloth and modern dependencies
!pip install --upgrade --no-cache-dir unsloth "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

## 1. Load Model & Tokenizer in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # Auto-detects bfloat16 on L4 / A100
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 2. Ingest Dataset
Upload `task_time_train.jsonl` and `task_time_eval.jsonl` (from `Fleet-Orchestrator/dataset/`) via the Colab files sidebar.

In [ ]:
import os
from datasets import load_dataset

# Determine filenames (supporting task_time or intent_routing naming)
train_file = "task_time_train.jsonl" if os.path.exists("task_time_train.jsonl") else "intent_routing_train.jsonl"
eval_file = "task_time_eval.jsonl" if os.path.exists("task_time_eval.jsonl") else "intent_routing_eval.jsonl"

dataset = load_dataset("json", data_files={
    "train": train_file,
    "eval": eval_file
})

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset['train'])} train samples, {len(dataset['eval'])} eval samples from {train_file}")

## 3. Train LoRA Adapters (Optimized for L4 / T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# L4 GPU has 24GB VRAM: use batch size 8 with bf16 for maximum speed
has_bf16 = torch.cuda.is_bf16_supported()
batch_size = 8 if has_bf16 else 4
grad_accum = 2 if has_bf16 else 4

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = batch_size,
        gradient_accumulation_steps = grad_accum,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not has_bf16,
        bf16 = has_bf16,
        logging_steps = 15,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 4. Test Structured Inference (Intent + Time Allocation)

In [ ]:
import json
FastLanguageModel.for_inference(model)

sys_prompt = "You are the high-speed Intent, Skill, and Task Time Allocation Router for the Aaradhya development ecosystem. Classify the incoming user intent into the exact lifecycle archetype, tier, 2D matrix cell, primary skill, supporting skills, velocity profile, and task time allocation (tier, estimated_duration_s, timeout_ceiling_s, cpm_weight, execution_route) in strict JSON format."

test_prompts = [
    "Fix the regression in test_warehouse_mem_sim.py where queue latency was calculating as zero.",
    "Scaffold IOE BE semester IV-II notes for CE 752 and EX 756 with syllabus markdown hubs.",
    "Dispatch autonomous fleet swarm across 27 workers to implement parallel unit tests.",
    "Add a new project card to access.js and re-encrypt with AES-256-GCM.",
    "Audit system memory and check git status."
]

for prompt in test_prompts:
    prompt_text = f"<|im_start|>system\n{sys_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt_text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 220, use_cache = True)
    res = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    print(f"\n{'='*70}")
    print(f"Prompt: {prompt}")
    try:
        parsed = json.loads(res)
        print(json.dumps(parsed, indent=2))
    except Exception:
        print(f"Raw Output:\n{res}")

## 5. Export Quantized INT4 GGUF (Q4_K_M) & Download
Quantizes the model directly to `q4_k_m` GGUF (~380 MB), ready to mount into LM Studio (`localhost:1234`) or llama.cpp.

In [ ]:
import glob
from google.colab import files

# Export quantized GGUF (~380 MB)
model.save_pretrained_gguf("qwen_intent_router_q4", tokenizer, quantization_method = "q4_k_m")

gguf_files = glob.glob("qwen_intent_router_q4/*.gguf")
if gguf_files:
    target_gguf = gguf_files[0]
    print(f"Quantized GGUF ready: {target_gguf}")
    print("Triggering browser download...")
    files.download(target_gguf)
else:
    print("Error: GGUF file not found in qwen_intent_router_q4/")

## (Optional) Direct Save to Google Drive
Mount your Google Drive to save the model directly without browser download delays.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
drive_dest = '/content/drive/MyDrive/models/'
os.makedirs(drive_dest, exist_ok=True)

if gguf_files:
    shutil.copy(gguf_files[0], drive_dest)
    print(f"Saved {gguf_files[0]} directly to {drive_dest}")